Система уравнений задана в виде $\frac{\partial u}{\partial t}+A\frac{\partial u}{\partial x}=0$, поскольку вектор правой части $b(x)$ нулевой.

Шаг 1: Поиск собственных значений матрицы A

Для приведения системы к характеристическому виду необходимо найти собственные значения $\lambda$ матрицы $A$. Решим характеристическое уравнение $\det(A - \lambda I) = 0$:$$\begin{vmatrix} 9-\lambda & -4 & -8 \\ -2 & 7-\lambda & -8 \\ 2 & 2 & 17-\lambda \end{vmatrix} = 0$$

Таким образом, собственные значения матрицы:

$\lambda_1 = 9$ 

$\lambda_2 = 11$

$\lambda_3 = 13$

Все собственные значения действительные и >0, значит, система является строго гиперболической.

Шаг 2: Поиск левых собственных векторов

Левые собственные векторы $l_i$ находятся из уравнения $A^T l_i = \lambda_i l_i$, где $A^T$ — транспонированная матрица:

$$A^T = \begin{pmatrix} 9 & -2 & 2 \\ -4 & 7 & 2 \\ -8 & -8 & 17 \end{pmatrix}$$

Шаг 3: Приведение к характеристическому видуСоставим матрицу перехода $S^{-1}$, строками которой являются левые собственные векторы:

$$S^{-1} = \begin{pmatrix} 0 & 1 & 1 \\ 1 & -1 & 0 \\ 1 & 0 & 2 \end{pmatrix}$$

Умножив исходную систему $\frac{\partial u}{\partial t}+A\frac{\partial u}{\partial x}=0$ слева на $S^{-1}$, мы получим систему независимых уравнений переноса (инвариантов Римана $w = S^{-1}u$):

$$w = S^{-1}u = \begin{pmatrix} 0 & 1 & 1 \\ 1 & -1 & 0 \\ 1 & 0 & 2 \end{pmatrix} \begin{pmatrix} u_1 \\ u_2 \\ u_3 \end{pmatrix} = \begin{pmatrix} u_2 + u_3 \\ u_1 - u_2 \\ u_1 + 2u_3 \end{pmatrix}$$

Система в характеристическом виде записывается как $\frac{\partial w}{\partial t} + \Lambda \frac{\partial w}{\partial x} = 0$, где $\Lambda = \text{diag}(9, 11, 13)$:

$$\frac{\partial (u_2 + u_3)}{\partial t} + 9 \frac{\partial (u_2 + u_3)}{\partial x} = 0$$

$$\frac{\partial (u_1 - u_2)}{\partial t} + 11 \frac{\partial (u_1 - u_2)}{\partial x} = 0$$
$$\frac{\partial (u_1 + 2u_3)}{\partial t} + 13 \frac{\partial (u_1 + 2u_3)}{\partial x} = 0$$

Шаг 4: Постановка граничных условий

Для корректной постановки граничных условий на отрезке $0 \le x \le 1$ необходимо проанализировать знаки собственных значений $\lambda_i$:

$\lambda_1 = 9 > 0$

$\lambda_2 = 11 > 0$

$\lambda_3 = 13 > 0$

Все собственные значения строго положительны, значит все 3 граничных условия должны на левой границе x=0:

$$\mathbf{u}(0, t) =\mathbf{\mu}(t)$$


## Схемы

### 1
$$\frac{u_{m}^{n+1} - u_{m}^n}{\tau} + a \frac{u_{m}^{n+1} - u_{m-1}^{n+1}}{h} = 0$$

$$u_{m}^{n+1} = \frac{1}{1 + c}u_{m}^n+\frac{c}{1 + c}u_{m-1}^{n+1}$$

### 2
$$\frac{u_{m}^{n+1}-u_{m}^{n}}{2\tau}+\frac{u_{m-1}^{n}-u_{m-1}^{n-1}}{2\tau}+a\frac{u_{m}^{n}-u_{m-1}^{n}}{h}=0$$

$$u_{m}^{n+1} = (1-2c)u_{m}^{n} +(2c-1)u_{m-1}^{n}+u_{m-1}^{n-1}$$


$$c = a \frac{\tau}{h}\text{ - число Куранта}$$



## Сходимость

Число куранта для 1-ой схемы может быть любым, т.к. она неявна (веса есть на слое n+1). 2 схема условно устойчивая. Для 2-ой схемы необходимо $c \leq 1$ (Условие Куранта), поэтому необходимо:

$M \leq \frac{N}{\lambda}$

In [ ]:
import numpy as np


L = 1.0
T = 1.0
# Собственные значения (из характеристического уравнения)
lambdas = np.array([9.0, 11.0, 13.0])

# Матрицы перехода
S_inv = np.array([[0, 1, 1], [1, -1, 0], [1, 0, 2]])

S = np.array([[2, 2, -1], [2, 1, -1], [-1, -1, 1]])


# Численное решение (скалярного уравнения)
def scalar(M_, N_, i, W_, scheme):
    tau_ = T / N_
    h_ = L / M_
    c = lambdas[i] * tau_ / h_
    for m in range(1, M_ + 1):
        W_[i, 1, m] = (W_[i, 0, m] + c * W_[i, 1, m - 1]) / (1 + c)

    for n in range(1, N_):
        # Идем по пространству слева направо

        for m in range(1, M_ + 1):
            if scheme == 1:
                W_[i, n + 1, m] = (W_[i, n, m] + c * W_[i, n + 1, m - 1]) / (1 + c)
            elif scheme == 2:
                W_[i, n + 1, m] = (
                    W_[i, n, m]
                    - (W_[i, n, m - 1] - W_[i, n - 1, m - 1])
                    - 2 * c * (W_[i, n, m] - W_[i, n, m - 1])
                )
    return W_


# Для вектора
def vect(M_, N_, W_, scheme):
    for i in range(3):
        W_ = scalar(M_, N_, i, W_, scheme)
    return W_


def ful(M_, N_, scheme):
    x = np.linspace(0, L, M_ + 1)
    # t = np.linspace(0, T, N_ + 1)

    # Начальные условия для u(x, 0)
    u = np.zeros((3, N_ + 1, M_ + 1))
    u[0, 0, :] = x**3
    u[1, 0, :] = 1 - x**2
    u[2, 0, :] = x**2 + 1

    # Граничные условия
    '''u[0, :, 0] = u[0, 0, 0]
    u[1, :, 0] = u[1, 0, 0]
    u[2, :, 0] = u[2, 0, 0]'''
    u[0, :, 0] = 0
    u[1, :, 0] = 1
    u[2, :, 0] = 0

    # Переход от u к w
    w = np.zeros((3, N_ + 1, M_ + 1))
    for n in range(N_ + 1):
        for m in range(M_ + 1):
            w[:, n, m] = S_inv @ u[:, n, m]

    w = vect(M_, N_, w, scheme)

    # Возврат к исходным переменным u
    u = np.zeros((3, N_ + 1, M_ + 1))
    for n in range(N_ + 1):
        for m in range(M_ + 1):
            u[:, n, m] = S @ w[:, n, m]

    return u

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import cm


def draw(M_, N_, scheme, index):

    # 2. Define your axes
    t = np.linspace(0, T, N_ + 1)  # Time axis
    x = np.linspace(0, L, M_ + 1)  # Space axis
    Time, X = np.meshgrid(t, x)  # Create the 2D grid

    # 3. Extract the 0th component
    # We transpose (.T) because meshgrid expects (Y, X) shape for the Z input
    u_res = ful(M_, N_, scheme)
    Z = u_res[index, :, :].T

    # 4. Plotting
    fig, ax = plt.subplots(subplot_kw={"projection": "3d"}, figsize=(10, 7))

    surf = ax.plot_surface(Time, X, Z, cmap=cm.viridis, linewidth=0, antialiased=False)

    # Labels and formatting
    ax.set_xlabel("Time (t)")
    ax.set_ylabel("Space (x)")
    ax.set_zlabel(f"{index}th Component Value")
    ax.set_title("3D Visualization of Vector Component")

    fig.colorbar(surf, shrink=0.5, aspect=5)
    plt.show()


draw(10, 1000, 2, 0)

## Первое дифференциальное приближение
Предварительно:
Из уравнения $u_t + a u_x = 0$ следует, что $u_t = -a u_x$. 

Продифференцировав это равенство, получим связь старших производных:

$u_{xt} = -a u_{xx}$, $\quad u_{tt} = a^2 u_{xx}$, $\quad u_{ttt} = -a^3 u_{xxx}$, $\quad u_{xxt} = -a u_{xxx}$, $\quad u_{xtt} = a^2 u_{xxx}$.

### 1 схема
Разложим сеточные значения функции в ряд Тейлора в узле $(x_m, t_{n+1})$.

$u_m^{n+1} = u$

$u_m^n = u - \tau u_t + \frac{\tau^2}{2} u_{tt} - O(\tau^3)$

$u_{m-1}^{n+1} = u - h u_x + \frac{h^2}{2} u_{xx} - O(h^3)$


Подставим разложения в разностную схему:

$$\frac{u - (u - \tau u_t + \frac{\tau^2}{2} u_{tt})}{\tau} + a \frac{u - (u - h u_x + \frac{h^2}{2} u_{xx})}{h} = 0$$

Упрощаем, оставляя главные члены ошибки:

$$(u_t - \frac{\tau}{2} u_{tt}) + a (u_x - \frac{h}{2} u_{xx}) = 0$$

$$u_t + a u_x = \frac{\tau}{2} u_{tt} + a \frac{h}{2} u_{xx}$$


Подставим $u_{tt}=a^2 u_{xx}$:

$$u_t + a u_x = \frac{\tau}{2} (a^2 u_{xx}) + a \frac{h}{2} u_{xx}$$

Получаем ПДП:

$$u_t + a u_x = \frac{ah}{2} (1 + c) u_{xx}$$



### 2 схема

Разложим сеточные значения функции в ряд Тейлора в узле $(x_m, t_n)$:


$u_{m-1}^n = u - h u_x + \frac{h^2}{2} u_{xx} - \frac{h^3}{6} u_{xxx} + O(h^4)$

$u_m^{n+1} = u + \tau u_t + \frac{\tau^2}{2} u_{tt} + \frac{\tau^3}{6} u_{ttt} + O(\tau^4)$

$u_{m-1}^{n-1} = u - h u_x - \tau u_t + \frac{h^2}{2} u_{xx} + \frac{\tau^2}{2} u_{tt} + h \tau u_{xt} - \frac{h^3}{6} u_{xxx} - \frac{\tau^3}{6} u_{ttt} - \frac{h^2 \tau}{2} u_{xxt} - \frac{h \tau^2}{2} u_{xtt} + \dots$

Подставим эти разложения в члены разностной схемы:

Производные по времени:

$$\frac{u_m^{n+1} - u_m^n}{2\tau} = \frac{1}{2} u_t + \frac{\tau}{4} u_{tt} + \frac{\tau^2}{12} u_{ttt}$$

$$\frac{u_{m-1}^n - u_{m-1}^{n-1}}{2\tau} = \frac{1}{2} u_t - \frac{\tau}{4} u_{tt} - \frac{h}{2} u_{xt} + \frac{\tau^2}{12} u_{ttt} + \frac{h^2}{4} u_{xxt} + \frac{h \tau}{4} u_{xtt}$$

Суммируя их, получаем:

$$u_t - \frac{h}{2} u_{xt} + \frac{\tau^2}{6} u_{ttt} + \frac{h^2}{4} u_{xxt} + \frac{h \tau}{4} u_{xtt}$$

Производная по пространству:

$$a\frac{u_m^n - u_{m-1}^n}{h} = a u_x - \frac{a h}{2} u_{xx} + \frac{a h^2}{6} u_{xxx}$$

Складываем всё вместе и группируем по порядку малости:

$$u_t + a u_x - \frac{h}{2}(u_{xt} + a u_{xx}) + \left( \frac{\tau^2}{6} u_{ttt} + \frac{h^2}{4} u_{xxt} + \frac{h \tau}{4} u_{xtt} + \frac{a h^2}{6} u_{xxx} \right) = 0$$


Подставляем $u_{xt} = -a u_{xx}$ в слагаемое порядка $O(h)$:

$$-\frac{h}{2}(-a u_{xx} + a u_{xx}) = 0$$

Это означает, что схема имеет второй порядок точности.

Выразим все слагаемые порядка $O(h^2, \tau^2)$ через $u_{xxx}$:

$$-\frac{a^3 \tau^2}{6} u_{xxx} - \frac{a h^2}{4} u_{xxx} + \frac{a^2 h \tau}{4} u_{xxx} + \frac{a h^2}{6} u_{xxx} = -a \left( \frac{a^2 \tau^2}{6} - \frac{a h \tau}{4} + \frac{h^2}{12} \right) u_{xxx}$$

Получаем ПДП:

$$\frac{\partial u}{\partial t} + a \frac{\partial u}{\partial x} = -a h^2 \frac{2c^2 - 3c + 1}{12} \frac{\partial^3 u}{\partial x^3}$$


## Диссипативная vs дисперсионная ошибка

1. Диссипативная ошибка

2. Дисперсионная ошибка

Если посмотреть на правую часть ПДП то всё становится на свои места

## Монотонность

Из определения монотонности  про Фридрихсу (коэффициенты разногстной схемы >=0):

1. Монотонна
2. Не монотонна, кроме случая $c=0.5$
## Апостериорная оценка порядка сходимости

Правило Рунге:

$$p \approx \log_2 \frac{\| \mathbf{u}^h - \mathbf{u}^{h/2} \|}{\| \mathbf{u}^{h/2} - \mathbf{u}^{h/4} \|}$$

Используем норму $L_\infty$:

$$||\mathbf{u}_m^n||_\infty=\max_{\substack{ m=0..M \\ n=0.. N}}|\mathbf{u}_m^n|$$

In [ ]:
def check_convergence_order(scheme):
    # Базовая сетка
    M1, N1 = 10, 10000
    # Сетка в 2 раза гуще
    M2, N2 = M1 * 2, N1 * 2
    # Сетка в 4 раза гуще
    M3, N3 = M1 * 4, N1 * 4

    u1 = ful(M1, N1, scheme)
    u2 = ful(M2, N2, scheme)
    u3 = ful(M3, N3, scheme)


    # Берем решения на последнем слое по времени (T = 1)
    # Чтобы сравнить векторы разной длины, извлекаем значения из густой сетки 
    # только в тех узлах, которые совпадают с базовой сеткой.
    u1_final = u1
    u2_final = u2[:, ::2, ::2]  # берем каждый 2-й узел
    u3_final = u3[:, ::4, ::4]  # берем каждый 4-й узел

    delta_u_1=u1_final-u2_final
    delta_u_2=u2_final-u3_final

    # Вычисляем L2-норму разности решений
    squared_length1=np.sum(delta_u_1**2,axis=0)
    length1=np.sqrt(squared_length1)
    norm1=np.max(length1)

    squared_length2=np.sum(delta_u_2**2,axis=0)
    length2=np.sqrt(squared_length2)
    norm2=np.max(length2)

   
    # Апостериорная оценка порядка по Рунге
    p = np.log2(norm1 / norm2)
    
    print(f"Схема {scheme}: Норма u_h - u_h/2 = {norm1:.6f}")
    print(f"Схема {scheme}: Норма u_h/2 - u_h/4 = {norm2:.6f}")
    print(f"Схема {scheme}: Апостериорный порядок сходимости p = {p:.2f}")
    
    return p

# Вызов функции для проверки второй схемы:
check_convergence_order(2)